# Solar Filament Segmentation Challenge 2026

## Reproducible, PQ-first Kaggle run

This notebook is deliberately executable end-to-end. It makes a submission only from a saved trained checkpoint and the competition test images. It contains no payload, hidden metadata, or precomputed test RLE.


## Legacy notebook audit

The prior notebook's claimed score cannot be attributed to its U-Net because its captured validation Dice was approximately zero while its output used a hard-coded precomputed-RLE route. It also used only 60 frames, five CPU epochs, destructive 256px crops, weak validation, no observation grouping, no instance/PQ objective, and no measured post-processing. This workflow removes that route and records only computed validation results.


In [ ]:
# Resolve the repository source in both a local checkout and Kaggle working directories.
from pathlib import Path
import sys

search_roots = [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working'), Path('/kaggle/input')]
repo_root = next((root for root in search_roots if (root / 'solar_filament').is_dir()), None)
if repo_root is None:
    candidates = list(Path('/kaggle/working').rglob('solar_filament')) if Path('/kaggle/working').exists() else []
    repo_root = candidates[0].parent if candidates else None
if repo_root is None:
    raise FileNotFoundError('Could not find solar_filament; attach or copy this repository to Kaggle working storage.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import solar_filament
from solar_filament.pipeline import PipelineConfig
print('Repository source:', repo_root)
print('solar_filament import:', solar_filament.__file__)


In [ ]:
import json
import logging
import time
from dataclasses import asdict

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch
from torch.utils.data import DataLoader

from solar_filament.pipeline import (
    FilamentSegmentationModel, SolarFilamentDataset, append_experiment, build_grad_scaler,
    build_instance_id_mask, build_instance_masks, build_semantic_mask, cache_validation_predictions,
    create_submission, environment_report, find_competition_data,
    group_train_validation_split, load_coco_annotations, measure_foreground_prevalence,
    optimize_postprocessing, run_synthetic_metric_tests, set_seed, train_one_epoch,
    validate,
)

# Final competition runs use all observations. DEBUG_MODE only exercises the pipeline quickly.
DEBUG_MODE = False
RUN_TRAINING = True
RUN_VALIDATION = True
RUN_TEST_INFERENCE = True
WORKING = Path('/kaggle/working')
WORKING.mkdir(parents=True, exist_ok=True)
config = PipelineConfig(
    debug_mode=DEBUG_MODE,
    epochs=2 if DEBUG_MODE else 50,
    max_train_images=None,
    train_tile_size=512 if DEBUG_MODE else 768,
    batch_size=1 if DEBUG_MODE else 2,
    inference_batch_size=1 if DEBUG_MODE else 4,
    use_tta=False,  # enable only after recording a validation improvement
)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
set_seed(config.seed)
print(json.dumps(environment_report(config), indent=2))
run_synthetic_metric_tests()  # perfect / FP / FN / merge / split / strict 0.5 checks
assert config.max_train_images is None


In [ ]:
data = find_competition_data('/kaggle/input')
index = load_coco_annotations(data.annotation_json)
train_ids, val_ids = group_train_validation_split(index, config.val_fraction, config.seed)
if DEBUG_MODE:
    train_ids, val_ids = train_ids[:8], val_ids[:4]
assert set(train_ids).isdisjoint(val_ids)
split_summary = {
    'physical_train_observations': len({Path(index.images[i]['file_name']).name for i in train_ids}),
    'physical_validation_observations': len({Path(index.images[i]['file_name']).name for i in val_ids}),
    'train_annotation_entries': len(train_ids),
    'validation_annotation_entries': len(val_ids),
}
assert split_summary['physical_train_observations'] and split_summary['physical_validation_observations']
prevalence = measure_foreground_prevalence(index, train_ids)
pos_weight = float(np.clip(prevalence['background_to_foreground'], 1.0, 50.0))
print(json.dumps({**prevalence, 'used_pos_weight': pos_weight}, indent=2))


## Annotation alignment sanity check

This generates 20 randomly selected training overlays before training. Inspect the output visually; do not train if polygons/RLE masks are offset or unexpectedly empty.


In [ ]:
rng = np.random.default_rng(config.seed)
visual_ids = rng.choice(train_ids, size=min(20, len(train_ids)), replace=False)
figure, axes = plt.subplots(4, 5, figsize=(20, 16))
for axis, image_id in zip(axes.flat, visual_ids):
    meta = index.images[image_id]
    image = np.asarray(Image.open(data.train_images / meta['file_name']).convert('L'))
    masks = build_instance_masks(index, image_id)
    semantic = build_semantic_mask(index, image_id)
    assert semantic.shape == image.shape and all(mask.shape == image.shape and mask.any() for mask in masks)
    assert all(mask.sum() > 0 for mask in masks)
    axis.imshow(image, cmap='gray')
    axis.imshow(semantic, cmap='Reds', alpha=.38)
    axis.set_title(f"entry={image_id}\n{Path(meta['file_name']).name}\ninstances={len(index.annotations_by_image.get(image_id, []))}")
    axis.axis('off')
for axis in axes.flat[len(visual_ids):]: axis.axis('off')
figure.tight_layout()
figure.savefig(WORKING / 'annotation_sanity_20.png', dpi=150)
plt.show()


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dataset = SolarFilamentDataset(index, data.train_images, train_ids, config, training=True)
val_dataset = SolarFilamentDataset(index, data.train_images, val_ids, config, training=False)
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,
                          num_workers=config.num_workers, pin_memory=device.type == 'cuda')
model = FilamentSegmentationModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
scaler = build_grad_scaler(device, config.use_amp)
print('Device:', device, 'parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
# Checkpoints are selected by full-image global PQ, not crop loss or training Dice.
best_path = WORKING / 'best_model.pth'
best_global_pq = -1.0
run_started = time.perf_counter()
if RUN_TRAINING:
    for epoch in range(config.epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, config, device, pos_weight)
        scheduler.step()
        metrics = validate(model, val_dataset, config, device)
        print({'epoch': epoch + 1, 'train_loss': train_loss, **metrics})
        if metrics['global_pq'] > best_global_pq:
            best_global_pq = metrics['global_pq']
            torch.save({'model': model.state_dict(), 'config': asdict(config), 'metrics': metrics, 'pos_weight': pos_weight}, best_path)
    print('Actual crop sampling distribution:', train_dataset.sampling_distribution())
if not best_path.exists():
    raise RuntimeError('No best checkpoint was saved; do not run validation/inference with an untrained model.')


In [ ]:
# Cache full-resolution validation maps once, then optimize only cached predictions.
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model'])
cache = cache_validation_predictions(model, val_dataset, config, device, WORKING / 'validation_cache')
if RUN_VALIDATION:
    optimized_config, search_table = optimize_postprocessing(
        [x[0] for x in cache], [x[1] for x in cache], [x[2] for x in cache], config,
    )
    search_table.to_csv(WORKING / 'postprocessing_search.csv', index=False)
    with (WORKING / 'optimized_config.json').open('w') as handle:
        json.dump(asdict(optimized_config), handle, indent=2)
    final_metrics = validate(model, val_dataset, optimized_config, device)
    print('Full-image validation (measured):', json.dumps(final_metrics, indent=2))
    print(search_table.to_string(index=False))
    append_experiment(WORKING / 'experiments.csv', 'resnet34_boundary', optimized_config, final_metrics,
                      time.perf_counter() - run_started, pos_weight, split_summary)
else:
    optimized_config, final_metrics = config, checkpoint['metrics']
with (WORKING / 'config.json').open('w') as handle:
    json.dump(asdict(optimized_config), handle, indent=2)
with (WORKING / 'validation_results.json').open('w') as handle:
    json.dump(final_metrics, handle, indent=2)


## Validation failure visualizations

The cached full-image predictions permit reviewing the largest split/merge/false-positive/fine-structure failures without retraining. The files are written to `/kaggle/working/debug/`.


In [ ]:
from solar_filament.pipeline import compute_pq, reconstruct_instances

debug_dir = WORKING / 'debug'; debug_dir.mkdir(exist_ok=True)
for image_id, (probability, boundary, truth) in zip(val_ids, cache):
    predicted = reconstruct_instances(probability, optimized_config, boundary)
    diagnostics = compute_pq(predicted, truth)
    failure_type = 'split' if diagnostics['one_to_many'] else 'merge' if diagnostics['many_to_one'] else 'false_positive' if diagnostics['fp'] else 'fine_structure' if diagnostics['fn'] else 'match'
    figure, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(probability, cmap='magma'); axes[0].set_title('semantic probability')
    axes[1].imshow(truth, cmap='nipy_spectral'); axes[1].set_title('GT instances')
    axes[2].imshow(predicted, cmap='nipy_spectral'); axes[2].set_title('predicted instances')
    axes[3].imshow((predicted > 0).astype(int) - (truth > 0).astype(int), cmap='bwr', vmin=-1, vmax=1); axes[3].set_title('FN / FP error')
    for axis in axes: axis.axis('off')
    figure.tight_layout(); figure.savefig(debug_dir / f'validation_{image_id}_{failure_type}.png', dpi=130); plt.close(figure)


In [ ]:
# The submission is generated solely from checkpoint logits + selected config + test images.
if RUN_TEST_INFERENCE:
    submission = create_submission(model, data, optimized_config, WORKING / 'submission.csv', device)
    assert list(submission.columns) == ['filament_id', 'segmentation_rle']
    print('Submission written:', WORKING / 'submission.csv')


## Reporting policy

The printed validation values are the only values this notebook reports. They are full-observation, held-out measurements; they are not leaderboard claims. Keep an ablation only when `global_pq` improves, and retain the generated `experiments.csv`, validation cache, configuration, model checkpoint, search table, and debug images with the submission.
